# agent_v1 on Kaggle

Runs `python -m chimera_agent.run` on a Kaggle T4 with Ollama, for the smoke cases and the full dev split.

Notebook settings: **GPU T4 x2**, **Internet on**, the private dataset `vladyslavsh0/chimera-agent-challenge-task-1` added as input, and the Kaggle secret `HF_TOKEN` (read token with access to `google/embeddinggemma-300m`) attached. Edit with Accelerator: None and switch to the GPU only to run.

Only `task1/agent_input` is passed to the agent. The dataset's `task1/labels` folder is never used here; scoring happens locally after downloading the run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!find /kaggle/input -maxdepth 6 -type d -name agent_input

Set the dataset path printed above and the commit to run. Use the commit that contains the code being evaluated, so `run_info.json` attributes the run to it.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

DATA_ROOT = "/kaggle/input/datasets/vladyslavsh0/chimera-agent-challenge-task-1"
COMMIT = "main"
REPO = "/kaggle/working/CHIMERA-Agent"
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
![ -d {REPO} ] || git clone -q https://github.com/VlaDisLav23232/CHIMERA-Agent.git {REPO}
%cd {REPO}
!git checkout -q {COMMIT} && git log -1 --format='%H %s'

Python 3.12 environment, the pinned baseline and the agent dependencies (same commands as the README).

In [ ]:
!pip install -q uv
!uv venv -q --python 3.12 .venv
!uv pip install -q --python .venv/bin/python -r requirements.txt
!bash scripts/setup_agent.sh

Ollama with the context length the runner checks, then the models (Gemma via Ollama, EmbeddingGemma via Hugging Face).

In [ ]:
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH": "16384"},
                          stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama --version

In [ ]:
!bash scripts/setup_agent.sh --models
!ollama list

Unit tests on Linux (tests that need local data skip here).

In [ ]:
!.venv/bin/python -m unittest discover -s tests -t .

A config copy that points `data_dir` at the dataset's agent inputs. Everything else stays as in `configs/agent.json`.

In [ ]:
import json
config = json.load(open("configs/agent.json"))
config["data_dir"] = f"{DATA_ROOT}/task1/agent_input"
json.dump(config, open("/kaggle/working/agent_kaggle.json", "w"), indent=2)
print(config)

Smoke: three dev cases outside the audit set. Check `run_info.json` and the per-case time before the full run.

In [ ]:
split = json.load(open("splits/task1.json"))
audit = set(json.load(open("splits/audit_task1.json"))["cases"])
SMOKE = [c for c in sorted(split["dev"]) if c not in audit][:3]
case_args = " ".join(f"--case {c}" for c in SMOKE)
!.venv/bin/python -m chimera_agent.run --name agent_v1_smoke --system agent_v1 {case_args} --config /kaggle/working/agent_kaggle.json
!.venv/bin/python -c "import json; i = json.load(open('results/agent_v1_smoke/run_info.json')); print(i['status'], {c: (r['status'], r.get('elapsed_seconds')) for c, r in i['cases'].items()})"

Full dev run (60 cases, about 1.5-2 GPU-hours).

In [ ]:
!.venv/bin/python -m chimera_agent.run --name agent_v1_dev --system agent_v1 --all-dev --config /kaggle/working/agent_kaggle.json

Zip a run without its index copy and download it right away (`/kaggle/working` is lost when the session ends). The zip contains patient information: keep it local, never commit or share it.

In [ ]:
RUN = "agent_v1_dev"
!cd results && zip -qr /kaggle/working/{RUN}.zip {RUN} -x "{RUN}/resources/*" && ls -lh /kaggle/working/{RUN}.zip